# 05 — Autoencoder za Anomaly Detection
**Cilj:** Koristiti Autoencoder kao **unsupervised** pristup detekciji prevara.

**Ideja:** Autoencoder treniramo **samo na legitimnim transakcijama**. Mreža uči da rekonstruiše normalne obrasce. Kada naiđe na fraud transakciju (anomaliju), rekonstrukcijska greška će biti veća → koristimo tu grešku kao anomaly score.

$$\text{AnomalyScore}(x) = ||x - \hat{x}||^2 \quad \text{(MSE rekonstrukcija)}$$

---

## 0. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json, os, warnings

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    classification_report, confusion_matrix,
    f1_score, roc_curve, precision_recall_curve
)

warnings.filterwarnings('ignore')
sns.set_theme(style='darkgrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

PROC    = '../data/processed/'
MODELS  = '../models/'
RESULTS = '../results/'

print(f'TensorFlow: {tf.__version__}')

## 1. Učitavanje podataka

In [ ]:
X_train = np.load(f'{PROC}X_train.npy')
X_val   = np.load(f'{PROC}X_val.npy')
X_test  = np.load(f'{PROC}X_test.npy')
y_train = np.load(f'{PROC}y_train.npy')
y_val   = np.load(f'{PROC}y_val.npy')
y_test  = np.load(f'{PROC}y_test.npy')

INPUT_DIM = X_train.shape[1]

# Autoencoder treniramo SAMO na legitimnim transakcijama
X_train_legit = X_train[y_train == 0]
X_val_legit   = X_val[y_val == 0]

print(f'Input dim: {INPUT_DIM}')
print(f'Train (samo legitimne): {X_train_legit.shape}')
print(f'Val   (samo legitimne): {X_val_legit.shape}')
print(f'Test  (sve):            {X_test.shape} | Fraud: {y_test.sum()}')

## 2. Arhitektura Autoencodera

```
Input(31) → Encoder: 64→32→16→8 (bottleneck) → Decoder: 8→16→32→64 → Output(31)
```

Bottleneck (latentni prostor) prisiljava mrežu da nauči kompaktnu reprezentaciju normalnih transakcija.

In [ ]:
def build_autoencoder(input_dim, encoding_dim=8, hidden_units=[64, 32, 16], dropout_rate=0.2):
    """
    Simetričan Autoencoder:
    - Encoder: kompresuje ulaz do latentnog prostora
    - Decoder: rekonstruiše ulaz iz latentnog prostora
    - Gubitak: MSE između ulaza i rekonstrukcije
    """
    # ---- ENCODER ----
    encoder_input = layers.Input(shape=(input_dim,), name='encoder_input')
    x = encoder_input
    
    for units in hidden_units:
        x = layers.Dense(units, kernel_initializer='he_normal')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        x = layers.Dropout(dropout_rate)(x)
    
    # Bottleneck — linearno, bez aktivacije
    bottleneck = layers.Dense(encoding_dim, name='bottleneck')(x)
    
    encoder = keras.Model(encoder_input, bottleneck, name='Encoder')
    
    # ---- DECODER ----
    decoder_input = layers.Input(shape=(encoding_dim,), name='decoder_input')
    x = decoder_input
    
    for units in reversed(hidden_units):
        x = layers.Dense(units, kernel_initializer='he_normal')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        x = layers.Dropout(dropout_rate)(x)
    
    reconstruction = layers.Dense(input_dim, activation='linear', name='reconstruction')(x)
    
    decoder = keras.Model(decoder_input, reconstruction, name='Decoder')
    
    # ---- AUTOENCODER ----
    ae_input = layers.Input(shape=(input_dim,), name='ae_input')
    encoded  = encoder(ae_input)
    decoded  = decoder(encoded)
    autoencoder = keras.Model(ae_input, decoded, name='Autoencoder')
    
    autoencoder.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss='mse'
    )
    return autoencoder, encoder, decoder


autoencoder, encoder, decoder = build_autoencoder(INPUT_DIM, encoding_dim=8)
autoencoder.summary()

## 3. Treniranje Autoencodera

In [ ]:
cb_ae = [
    callbacks.EarlyStopping(monitor='val_loss', mode='min', patience=10,
                             restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor='val_loss', mode='min',
                                 factor=0.5, patience=5, min_lr=1e-6, verbose=1),
    callbacks.ModelCheckpoint(f'{MODELS}autoencoder.keras', monitor='val_loss',
                               mode='min', save_best_only=True)
]

# Ulaz = Izlaz = legitimne transakcije (rekonstrukcija)
history_ae = autoencoder.fit(
    X_train_legit, X_train_legit,
    validation_data=(X_val_legit, X_val_legit),
    epochs=150,
    batch_size=256,
    callbacks=cb_ae,
    verbose=1
)

# History plot
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(history_ae.history['loss'], label='Train MSE')
ax.plot(history_ae.history['val_loss'], label='Val MSE')
ax.set_title('Autoencoder — Rekonstrukcijska Greška (MSE)')
ax.set_xlabel('Epoha')
ax.set_ylabel('MSE Loss')
ax.legend()
plt.savefig(f'{RESULTS}05_ae_history.png', bbox_inches='tight')
plt.show()

## 4. Anomaly Score = Rekonstrukcijska Greška

In [ ]:
def reconstruction_error(model, X):
    """MSE po uzorku između originala i rekonstrukcije."""
    X_reconstructed = model.predict(X, verbose=0)
    mse = np.mean(np.power(X - X_reconstructed, 2), axis=1)
    return mse


# Anomaly scores na test setu
test_mse = reconstruction_error(autoencoder, X_test)

# Distribucija greške po klasama
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
legit_mse = test_mse[y_test == 0]
fraud_mse = test_mse[y_test == 1]

axes[0].hist(legit_mse, bins=100, alpha=0.6, color='#2ecc71', label='Legitimna', density=True)
axes[0].hist(fraud_mse, bins=100, alpha=0.6, color='#e74c3c', label='Fraud', density=True)
axes[0].set_xlabel('Rekonstrukcijska Greška (MSE)')
axes[0].set_ylabel('Gustina')
axes[0].set_title('Distribucija Anomaly Score po Klasama')
axes[0].set_xlim(0, np.percentile(test_mse, 99.5))
axes[0].legend()

# Log skala
axes[1].hist(legit_mse, bins=100, alpha=0.6, color='#2ecc71', label='Legitimna', density=True)
axes[1].hist(fraud_mse, bins=100, alpha=0.6, color='#e74c3c', label='Fraud', density=True)
axes[1].set_xlabel('Rekonstrukcijska Greška (MSE)')
axes[1].set_ylabel('Gustina (log)')
axes[1].set_title('Distribucija Anomaly Score (log osa)')
axes[1].set_yscale('log')
axes[1].legend()

plt.tight_layout()
plt.savefig(f'{RESULTS}05_anomaly_score_dist.png', bbox_inches='tight')
plt.show()

print(f'\nProsečna MSE — Legitimna: {legit_mse.mean():.4f}')
print(f'Prosečna MSE — Fraud:     {fraud_mse.mean():.4f}')
print(f'Separation ratio:          {fraud_mse.mean() / legit_mse.mean():.2f}x')

## 5. Određivanje Threshold-a i Evaluacija

In [ ]:
# Threshold na Val setu
val_mse = reconstruction_error(autoencoder, X_val)

thresholds = np.percentile(val_mse, np.linspace(80, 99.9, 200))
f1s = [f1_score(y_val, (val_mse >= t).astype(int)) for t in thresholds]
best_t_idx = np.argmax(f1s)
best_t = thresholds[best_t_idx]

print(f'Optimalni threshold (percentila na val): {best_t:.4f}')
print(f'Odgovarajuća F1: {max(f1s):.4f}')

# Vizualizacija F1 vs threshold
plt.figure(figsize=(9, 4))
plt.plot(thresholds, f1s, color='steelblue')
plt.axvline(best_t, color='red', linestyle='--', label=f'Optimalni threshold = {best_t:.3f}')
plt.xlabel('Threshold (MSE)')
plt.ylabel('F1 Score')
plt.title('F1 Score vs Anomaly Threshold')
plt.legend()
plt.savefig(f'{RESULTS}05_threshold_f1.png', bbox_inches='tight')
plt.show()

In [ ]:
# Finalna evaluacija na test setu
y_pred_ae = (test_mse >= best_t).astype(int)

roc_auc = roc_auc_score(y_test, test_mse)
pr_auc  = average_precision_score(y_test, test_mse)
f1      = f1_score(y_test, y_pred_ae)

print('=== AUTOENCODER — Finalna Evaluacija ===')
print(f'ROC-AUC: {roc_auc:.4f}')
print(f'PR-AUC:  {pr_auc:.4f}')
print(f'F1:      {f1:.4f}')
print()
print(classification_report(y_test, y_pred_ae, target_names=['Legitimna', 'Fraud']))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

cm = confusion_matrix(y_test, y_pred_ae)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Legit', 'Fraud'], yticklabels=['Legit', 'Fraud'])
axes[0].set_title('Confusion Matrix')
axes[0].set_ylabel('Stvarno')
axes[0].set_xlabel('Predviđeno')

fpr, tpr, _ = roc_curve(y_test, test_mse)
axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'AUC = {roc_auc:.4f}')
axes[1].plot([0, 1], [0, 1], 'k--')
axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
axes[1].set_title('ROC Kriva'); axes[1].legend()

prec, rec, _ = precision_recall_curve(y_test, test_mse)
axes[2].plot(rec, prec, color='steelblue', lw=2, label=f'PR-AUC = {pr_auc:.4f}')
axes[2].set_xlabel('Recall'); axes[2].set_ylabel('Precision')
axes[2].set_title('PR Kriva'); axes[2].legend()

plt.suptitle('Autoencoder Anomaly Detection', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{RESULTS}05_ae_evaluation.png', bbox_inches='tight')
plt.show()

# Čuvamo rezultate
ae_results = {'title': 'Autoencoder', 'roc_auc': roc_auc, 'pr_auc': pr_auc, 'f1': f1, 'threshold': best_t}
pd.DataFrame([ae_results]).to_csv(f'{RESULTS}05_ae_results.csv', index=False)

## 6. Vizualizacija Latentnog Prostora (t-SNE)

In [ ]:
from sklearn.manifold import TSNE

# Uzmemo sample radi brzine
n_sample = 3000
idx = np.random.choice(len(X_test), n_sample, replace=False)
X_sample = X_test[idx]
y_sample = y_test[idx]

# Enkodujemo u latentni prostor
latent = encoder.predict(X_sample, verbose=0)

print('Pokrećem t-SNE (može trajati par minuta)...')
tsne = TSNE(n_components=2, random_state=RANDOM_SEED, perplexity=30, n_iter=1000)
latent_2d = tsne.fit_transform(latent)

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#2ecc71' if y == 0 else '#e74c3c' for y in y_sample]
sizes  = [8 if y == 0 else 60 for y in y_sample]
alphas = [0.3 if y == 0 else 0.9 for y in y_sample]

for i in range(len(latent_2d)):
    ax.scatter(latent_2d[i, 0], latent_2d[i, 1],
               c=colors[i], s=sizes[i], alpha=alphas[i])

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#2ecc71', label='Legitimna'),
    Patch(facecolor='#e74c3c', label='Fraud')
]
ax.legend(handles=legend_elements, fontsize=12)
ax.set_title('t-SNE vizualizacija Latentnog Prostora Autoencodera', fontsize=13, fontweight='bold')
ax.set_xlabel('t-SNE Dimenzija 1')
ax.set_ylabel('t-SNE Dimenzija 2')

plt.tight_layout()
plt.savefig(f'{RESULTS}05_tsne_latent.png', bbox_inches='tight')
plt.show()

print('\n→ Sledeći notebook: 06_evaluation.ipynb')